# Alphamind新手入门之六：公式化选股

In [ ]:
import asyncio
import datetime as dt
import numpy as np


## 1. 公式书写
-------------------------

In [ ]:
# CSTopN 横截面选取Top N个值
# 选取申万一级每个行业中，ROE因子值最高的前3个
# 具体的公式化函数可以参考融量官方说明文档
import pyarrow as pa
import pyarrow.compute as pc

rng = np.random.default_rng(42)


## 2. 获取数据
---------------

In [ ]:
n_dates = 6
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]


In [ ]:
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets


## 3. Stock Screening
---------------------

In [ ]:
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.data.transformer import Transformer
from alphamind.formula import CSBottomN, CSRank, CSTopN


In [ ]:
features = await asyncio.to_thread(
    Transformer({
        "rank": CSRank("factor"),
        "top": CSTopN("factor", 3),
        "bottom": CSBottomN("factor", 3),
    }).transform,
    "code",
    panel,
)
result = pa.table({"trade_date": panel["trade_date"], "code": panel["code"]})
for name in features.column_names:
    result = result.append_column(name, features[name])


In [ ]:
selected = result.filter(pc.cast(result["top"], pa.bool_()))
assert selected.num_rows == n_dates * 3
selected
